![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Proyecto 2 - Clasificación de género de películas

El propósito de este proyecto es que puedan poner en práctica, en sus respectivos grupos de trabajo, sus conocimientos sobre técnicas de preprocesamiento, modelos predictivos de NLP, y la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 2: Clasificación de género de películas"

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 8. Sin embargo, es importante que avancen en la semana 7 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 8, y subir el archivo de predicciones a la [competencia de Kaggle](https://www.kaggle.com/t/29c44fce98c747f2a1dfdaf29d4c4965).

## Datos para la predicción de género en películas

![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/moviegenre.png)

En este proyecto se usará un conjunto de datos de géneros de películas. Cada observación contiene el título de una película, su año de lanzamiento, la sinopsis o plot de la película (resumen de la trama) y los géneros a los que pertenece (una película puede pertenercer a más de un género). Por ejemplo:
- Título: 'How to Be a Serial Killer'
- Plot: 'A serial killer decides to teach the secrets of his satisfying career to a video store clerk.'
- Generos: 'Comedy', 'Crime', 'Horror'

La idea es que usen estos datos para predecir la probabilidad de que una película pertenezca, dada la sinopsis, a cada uno de los géneros.

Agradecemos al profesor Fabio González, Ph.D. y a su alumno John Arevalo por proporcionar este conjunto de datos. Ver https://arxiv.org/abs/1702.01992

## Ejemplo predicción conjunto de test para envío a Kaggle
En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Importación librerías
import pandas as pd
import os
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [3]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip', encoding='UTF-8', index_col=0)

In [ ]:
# Visualización datos de entrenamiento
dataTraining.head()

,year,title,plot,genres,rating
3107,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0
900,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6
6724,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2
4704,1954,Executive Suite,"in a friday afternoon in new york , the presi...",['Drama'],7.4
2582,1990,Narrow Margin,"in los angeles , the editor of a publishing h...","['Action', 'Crime', 'Thriller']",6.6


In [5]:
# Visualización datos de test
dataTesting.head()

,year,title,plot
1,1999,Message in a Bottle,"who meets by fate , shall be sealed by fate ...."
4,1978,Midnight Express,"the true story of billy hayes , an american c..."
5,1996,Primal Fear,martin vail left the chicago da ' s office to ...
6,1950,Crisis,husband and wife americans dr . eugene and mr...
7,1959,The Tingler,the coroner and scientist dr . warren chapin ...


In [6]:
# Definición de variables predictoras (X)
vect = CountVectorizer(max_features=1000)
X_dtm = vect.fit_transform(dataTraining['plot'])
X_dtm.shape

(7895, 1000)

In [7]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
le = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining['genres'])

In [8]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train, X_test, y_train_genres, y_test_genres = train_test_split(X_dtm, y_genres, test_size=0.33, random_state=42)

In [9]:
# Definición y entrenamiento
clf = OneVsRestClassifier(RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=10, random_state=42))
clf.fit(X_train, y_train_genres)

,estimator,RandomForestC...ndom_state=42)
,n_jobs,None
,verbose,0
,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None


In [10]:
# Predicción del modelo de clasificación
y_pred_genres = clf.predict_proba(X_test)

# Impresión del desempeño del modelo
roc_auc_score(y_test_genres, y_pred_genres, average='macro')

0.7717668029296991

In [11]:
# transformación variables predictoras X del conjunto de test
X_test_dtm = vect.transform(dataTesting['plot'])

cols = ['p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy', 'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical', 'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport', 'p_Thriller', 'p_War', 'p_Western']

# Predicción del conjunto de test
y_pred_test_genres = clf.predict_proba(X_test_dtm)

In [12]:
# Guardar predicciones en formato exigido en la competencia de kaggle
res = pd.DataFrame(y_pred_test_genres, index=dataTesting.index, columns=cols)
res.to_csv('pred_genres_text_RF.csv', index_label='ID')
res.head()

,p_Action,p_Adventure,p_Animation,p_Biography,p_Comedy,p_Crime,p_Documentary,p_Drama,p_Family,p_Fantasy,...,p_Musical,p_Mystery,p_News,p_Romance,p_Sci-Fi,p_Short,p_Sport,p_Thriller,p_War,p_Western
1,0.115316,0.112545,0.021632,0.030826,0.396257,0.116158,0.062036,0.495677,0.066617,0.104482,...,0.025148,0.097838,0.000010,0.392608,0.064968,0.011944,0.018122,0.203969,0.022146,0.019669
4,0.133081,0.087854,0.023964,0.070760,0.343026,0.202546,0.099520,0.511076,0.063784,0.062972,...,0.024627,0.061687,0.002156,0.147331,0.059040,0.012346,0.019008,0.199905,0.040175,0.020344
5,0.158209,0.140732,0.018376,0.067342,0.340637,0.469086,0.006762,0.604303,0.088218,0.103984,...,0.022971,0.334849,0.000000,0.387805,0.112995,0.015059,0.022805,0.392327,0.067718,0.018431
6,0.165241,0.139772,0.027845,0.093197,0.333114,0.146278,0.020983,0.591203,0.076191,0.068679,...,0.111233,0.117495,0.010000,0.186832,0.099408,0.001430,0.038273,0.290187,0.079881,0.018423
7,0.165107,0.177139,0.046025,0.031550,0.317075,0.213078,0.028251,0.463566,0.077800,0.139171,...,0.024349,0.076296,0.000000,0.184966,0.266775,0.004899,0.020132,0.229698,0.023483,0.016585


Punto 1

In [13]:
# Librerías
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
import ast

# Variable predictora
X = dataTraining['plot']

# Convertir genres solo si viene como texto
def convertir_generos(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

dataTraining['genres'] = dataTraining['genres'].apply(convertir_generos)

# Variable objetivo
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(dataTraining['genres'])

# División entrenamiento y validación
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Vectorización
vect = TfidfVectorizer(
    stop_words='english',
    max_features=5000,
    ngram_range=(1,2)
)

X_train_dtm = vect.fit_transform(X_train)
X_val_dtm = vect.transform(X_val)

# Modelo
clf = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
)

clf.fit(X_train_dtm, y_train)

# Predicción validación
y_pred_val_genres = clf.predict_proba(X_val_dtm)

# AUC
auc = roc_auc_score(y_val, y_pred_val_genres, average='macro')
print("ROC-AUC:", auc)

ROC-AUC: 0.8098852060309603


In [14]:
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score

# Usar título + sinopsis
X = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))
X_kaggle = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('features', FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            stop_words='english',
            analyzer='word',
            ngram_range=(1, 2),
            max_features=50000,
            min_df=2,
            sublinear_tf=True
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            max_features=30000,
            min_df=2,
            sublinear_tf=True
        ))
    ])),
    ('model', OneVsRestClassifier(
        LogisticRegression(
            C=4,
            solver='liblinear',
            max_iter=1000,
            random_state=42
        )
    ))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict_proba(X_val)

auc = roc_auc_score(y_val, y_pred, average='macro')
print("ROC-AUC:", auc)

ROC-AUC: 0.901475256759556


In [15]:
y_test_pred = pipeline.predict_proba(X_kaggle)

cols = ['p_' + genre for genre in mlb.classes_]

submission = pd.DataFrame(
    y_test_pred,
    index=dataTesting.index,
    columns=cols
)

submission.to_csv('submission_proyecto2_mejorado.csv', index_label='ID')

submission.head()


,p_Action,p_Adventure,p_Animation,p_Biography,p_Comedy,p_Crime,p_Documentary,p_Drama,p_Family,p_Fantasy,...,p_Musical,p_Mystery,p_News,p_Romance,p_Sci-Fi,p_Short,p_Sport,p_Thriller,p_War,p_Western
1,0.063742,0.031827,0.009125,0.009521,0.146086,0.049483,0.008629,0.524957,0.018003,0.094036,...,0.024296,0.035825,0.000869,0.818623,0.009412,0.005450,0.011819,0.089247,0.006138,0.018814
4,0.084916,0.006643,0.019239,0.186347,0.221185,0.414073,0.029240,0.922718,0.008393,0.006870,...,0.014649,0.007968,0.001694,0.012547,0.002748,0.009164,0.011232,0.151458,0.021508,0.006043
5,0.027685,0.003959,0.001482,0.056689,0.020496,0.835162,0.009067,0.817680,0.001978,0.006204,...,0.007398,0.461314,0.001015,0.075001,0.022164,0.001510,0.008441,0.623441,0.009996,0.006274
6,0.063767,0.055394,0.002900,0.030371,0.062156,0.014833,0.017547,0.869179,0.007429,0.024523,...,0.013913,0.050700,0.000783,0.195035,0.050253,0.002184,0.010528,0.459738,0.075221,0.008621
7,0.019596,0.026629,0.021106,0.014244,0.207493,0.029799,0.009691,0.347289,0.023408,0.059373,...,0.008378,0.027706,0.000767,0.065472,0.642284,0.006191,0.003795,0.192020,0.006110,0.005067


import joblib

joblib.dump(pipeline, 'pipeline_generos.pkl')
joblib.dump(mlb, 'mlb_generos.pkl')

print("Artefactos guardados")